<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Marcelo Palma M.
- Nombre de alumno 2: Isaac Torrejon O.


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/Mpalma325/Laboratorio-de-Programacion-Cientifica-para-Ciencia-de-Datos.git)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [22]:
!pip install -qq xgboost optuna

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [23]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Lab6')
print(f"Current working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/MyDrive/Lab6


In [24]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv("./sales.csv")

df.head()

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


In [25]:
df.dtypes

,0
id,int64
date,object
city,object
lat,float64
long,float64
pop,int64
shop,object
brand,object
container,object
capacity,object


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [26]:
from sklearn import set_config
set_config(transform_output="pandas")

# Inserte su código acá

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# 1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%).
X = df.drop(['quantity','id'], axis=1)
y = df['quantity']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=7202)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=7202)

# 2. Implemente un FunctionTransformer para extraer el día, mes y año de la variable date.
def extract_date_features(X):
    X_copy = X.copy()
    X_copy['date'] = pd.to_datetime(X_copy['date'], format='%d/%m/%y')
    X_copy['year'] = X_copy['date'].dt.year.astype('category')
    X_copy['month'] = X_copy['date'].dt.month.astype('category')
    X_copy['day'] = X_copy['date'].dt.day.astype('category')
    return X_copy.drop('date', axis=1)

date_transformer = FunctionTransformer(extract_date_features)
date_transformer.set_output(transform="pandas")

# 3. Implemente un ColumnTransformer para procesar de manera adecuada los datos numéricos y categóricos.
categorical_features = ['city', 'shop', 'brand', 'container', 'capacity', 'year', 'month', 'day']
numerical_features = ['lat', 'long', 'pop', 'price']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features) # Set sparse_output=False
    ],
    remainder='passthrough'
)

preprocessor.set_output(transform='pandas')

# 4. Guarde los pasos anteriores en un Pipeline, dejando como último paso el regresor DummyRegressor.
pipeline_dummy = Pipeline(steps=[
    ('date_features', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy="mean"))
])

# 5. Entrene el pipeline anterior y reporte la métrica mean_absolute_error sobre los datos de validación.
pipeline_dummy.fit(X_train, y_train)
y_pred_dummy = pipeline_dummy.predict(X_val)
mae_dummy = mean_absolute_error(y_val, y_pred_dummy)

print(f"MAE DummyRegressor: {mae_dummy}")

# 6. Vuelva a entrenar el Pipeline pero esta vez usando XGBRegressor como modelo.
pipeline_xgb = Pipeline(steps=[
    ('date_features', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=7202))
])

pipeline_xgb.fit(X_train, y_train)
y_pred_xgb = pipeline_xgb.predict(X_val)
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)

print(f"MAE XGBRegressor (Parametros por default): {mae_xgb}")

# 7. Guarde ambos modelos en un archivo .pkl
joblib.dump(pipeline_dummy, 'dummy_regressor_pipeline.pkl')
joblib.dump(pipeline_xgb, 'xgb_regressor_pipeline_default.pkl')


MAE DummyRegressor: 13408.357159370607
MAE XGBRegressor (Parametros por default): 2445.652587890625


['xgb_regressor_pipeline_default.pkl']

El valor de MAE DummyRegressor indica que, en promedio, el modelo se equivoca en 13408 unidades por observación. Es decir, la predicción de demanda difiere en esa cantidad con respecto al valor real en cada caso.

Luego, el valor de MAE XGBRegressor indica que ahora el modelo , en promedio, el modelo se equivoca en 2445 unidades por observación, mejorando significativamente y siendo mejor que DummyRegressor ya que logra predecir con mayor precision la demanda


## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [27]:
# Inserte su código acá

# 1. Volver a entrenar el Pipeline con XGBRegressor, pero forzando una relación monótona negativa entre el precio y la cantidad.
feature_names = pipeline_xgb.named_steps['preprocessor'].get_feature_names_out()

monotonic_constraints = []
for name in feature_names:
    if "price" in name:
        monotonic_constraints.append(-1)
    else:
        monotonic_constraints.append(0)

monotonic_constraints = tuple(monotonic_constraints)

constrained_pipeline = Pipeline([
    ("date_features", date_transformer),
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(monotone_constraints=monotonic_constraints, random_state=7202))
])

constrained_pipeline.fit(X_train, y_train)

# 2. Reportar el MAE sobre el conjunto de validación
y_pred2 = constrained_pipeline.predict(X_val)
mae_constrained = mean_absolute_error(y_val, y_pred2)
print(f"MAE con restricción monótona: {mae_constrained:.2f}")

# 3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo?
print(f"MAE sin restricción:{mae_xgb:.2f}")
print(f"Diferencia: {mae_constrained - mae_xgb:.2f}")

# 4. Guarde su modelo en un archivo .pkl
joblib.dump(constrained_pipeline, "constrained_model.pkl")

MAE con restricción monótona: 2459.38
MAE sin restricción:2445.65
Diferencia: 13.73


['constrained_model.pkl']

Al imponer la restricción monótona negativa entre el precio y la cantidad demandada, el MAE aumentó levemente en 13.73 unidades, indicando que el modelo con restricción tiene un desempeño predictivo muy similar al modelo sin restricción, aunque minimamente menos preciso.

In [65]:
from scipy.stats import spearmanr

rho_s, p_s = spearmanr(df["price"], df["quantity"])
print(f"Correlación: {rho_s:.3f}")

Correlación: -0.620


A pesar del pequeño aumento en el error, la diferencia es mínima, se ve que la relación negativa entre el precio y la demanda ya estaba presente en los datos. Al imponer formalmente la restricción, el modelo mantiene casi la misma capacidad predictiva. Por lo tanto, el colega tenía razón al afirmar que la demanda guarda una relación inversa con el precio. Finalmente, de forma adicional, el valor de la correlación confirma lo expuesto por el colega, reafirmando la veracidad de lo dicho por el.

## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [29]:
import optuna
from optuna.samplers import TPESampler
import random
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Inserte su código acá

def objective(trial):
    # Inserte su código acá

    # Hiperparámetros:
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0)
    min_frequency = trial.suggest_float("min_frequency", 0.0, 1.0)

    # Preprocesamiento
    onehot_encoder = OneHotEncoder(min_frequency=min_frequency, sparse_output=False, handle_unknown='ignore')

    updated_preprocessor = ColumnTransformer([
        ("cat", onehot_encoder, categorical_features),
        ("num", "passthrough", numerical_features)
    ])

    model = XGBRegressor(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=7202
    )

    pipe = Pipeline([
        ("date_features", date_transformer),
        ("preprocessor", updated_preprocessor),
        ("regressor", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    trial.set_user_attr("model", pipe)
    return mae

# 2. Fijar tiempo de entrenamiento a 5 minutos
study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=7202))
study.optimize(objective, timeout=300, show_progress_bar=True)

# 3. Optimizar el modelo y reportar el número de trials, el MAE y los mejores hiperparámetros encontrados.
best_trial = study.best_trial
print(f"Trials: {len(study.trials)}")
print(f"Mejor MAE: {best_trial.value:.4f}")
print("Mejores hiperparámetros:")
for k, v in best_trial.params.items():
    print(f"  {k}: {v}")

   0%|          | 00:00/05:00

Trials: 163
Mejor MAE: 1987.3025
Mejores hiperparámetros:
  learning_rate: 0.07370935291796726
  n_estimators: 758
  max_depth: 9
  max_leaves: 92
  min_child_weight: 5
  reg_alpha: 0.24426480487635963
  reg_lambda: 0.3762319537168008
  min_frequency: 0.019410286882341733


Podemos notar que, con respecto a la seccion anterior, el valor de MAE disminuye a, logrando predecir con mayor precision la demanda. Esto ocurre debido a que Optuna permite elegir los mejores hiperparámetros para minimizar el MAE, naturalmente haciendolo un mejor modelo.  

- **learning_rate (0.001, 0.1)**: Controla la velocidad de aprendizaje del modelo. Cuanto menor, más lento pero preciso el ajuste. Se utiliza un rango conservador típico para evitar sobreajuste.

- **n_estimators (50, 1000)**: Número de árboles (boosting rounds) que utiliza el modelo. Más árboles pueden capturar mejor patrones, pero también tardan más. Tiene un rango razonable que cubre modelos rápidos y modelos más complejos.

- **max_depth (3, 10)**: Indica la profundidad máxima de cada árbol. Más profundidad implica más complejidad. El rango de 3 a 10 entrega un buen compromiso entre underfitting y overfitting.

- **max_leaves (0, 100)**: Se refiere al número máximo de hojas en los árboles del modelo. El rango tiene sentido llegue hasta 100 para no hacer un modelo demasiado complejo.

- **min_child_weight (1, 5)**: Este parámetro indica la mínima suma de pesos de observaciones necesarias para dividir un nodo. Un valor más alto indica un modelo más conservador. El rango de 1 a 5 permite evitar splits por ruido.

- **reg_alpha (0, 1)**: Este parámetro penaliza por L1 (sparse weights). El rango ayuda a la regularización y a reducir overfitting en el modelo.

- **reg_lambda (0, 1)**: Este penaliza por L2. Ayuda a controlar la magnitud de los coeficientes. El rango tiene sentido, pero se podrian explorar valores mayores para ver que ocurre.

- **min_frequency (0, 1)**: Usado para el one hot encoder, computa la minima frecuencia en el dataset que debe tener una categoria para ser considerada. El rango cubre todas las posibilidades por lo que tiene sentido.

In [30]:
# 5. Guardar el mejor modelo en un archivo .pkl
best_model = best_trial.user_attrs["model"]
joblib.dump(best_model, "best_xgb_model.pkl")

['best_xgb_model.pkl']

## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [31]:
!pip install optuna-integration[xgboost]

Prunning es un mecanismo para interrumpir tempranamente las búsquedas de hiperparámetros que muestran un "mal" desempeño con respecto al resto, al mismo punto del entrenamiento. El objetivo es ahorrar tiempo y recursos computacionales, modifcando drásticamente el tiempo total de la optimización pero sin hacerlo en la calidad del mejor modelo final.

In [47]:
import optuna
from optuna.samplers import TPESampler
from optuna.integration import XGBoostPruningCallback
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 2. Redefinir la función objective() utilizando optuna.integration.XGBoostPruningCallback como método de Prunning
def objective_prune(trial):
    # Hiperparámetros
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "max_leaves": trial.suggest_int("max_leaves", 0, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
        "seed": 7202,
        "verbosity": 0,
    }
    n_rounds = trial.suggest_int("n_estimators", 50, 1000)
    min_freq = trial.suggest_float("min_frequency", 0.0, 1.0)
    ohe = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        min_frequency=min_freq
    )

    preproc = ColumnTransformer([
        ("cat", ohe, categorical_features),
        ("num", "passthrough", numerical_features)
    ]).set_output(transform="pandas")

    Xdt_tr = date_transformer.transform(X_train)
    Xdt_vl = date_transformer.transform(X_val)

    Xtr = preproc.fit_transform(Xdt_tr)
    Xvl = preproc.transform(Xdt_vl)

    dtrain = xgb.DMatrix(Xtr, label=y_train)
    dvalid = xgb.DMatrix(Xvl, label=y_val)

    # Callback de pruning
    pruning_cb = XGBoostPruningCallback(trial, "validation-mae")

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=n_rounds,
        evals=[(dvalid, "validation")],
        callbacks=[pruning_cb],
        verbose_eval=False
    )

    preds = booster.predict(dvalid)
    mae = mean_absolute_error(y_val, preds)

    trial.set_user_attr("model", (preproc, booster))
    trial.set_user_attr("val_mae", float(mae))
    return mae

# 3. Fijar nuevamente el tiempo de entrenamiento a 5 minutos
study_prune = optuna.create_study(direction="minimize", sampler=TPESampler(seed=7202), pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
study_prune.optimize(objective_prune, timeout=300, show_progress_bar=True)

# 4. Reportar el número de trials, el MAE y los mejores hiperparámetros encontrados
best = study_prune.best_trial
print(f"Trials completados: {len(study_prune.trials)}")
print(f"Mejor MAE con pruning: {best.value:.6f}")
print("Hiperparámetros óptimos:")
for k, v in best.params.items():
    print(f"  {k}: {v}")

# 5. Guardar el mejor modelo en un archivo .pkl
best_model = best.user_attrs["model"]
joblib.dump(best_model, "best_xgb_modelPRU.pkl")

   0%|          | 00:00/05:00

Trials completados: 207
Mejor MAE con pruning: 1961.734741
Hiperparámetros óptimos:
  learning_rate: 0.09440138349488773
  max_depth: 10
  max_leaves: 87
  min_child_weight: 5
  reg_alpha: 0.6361346354266787
  reg_lambda: 0.480662979102613
  n_estimators: 984
  min_frequency: 0.04378788144627266


['best_xgb_modelPRU.pkl']

Al comparar los resultados de la optimización con y sin prunning, notamos que el MAE con prunning es ligeramente mejor que el de sin, indicando que, la implementación logró encontrar un modelo con un rendimiento predictivo superior, para el mismo tiempo de búsqueda. Esto se debe a que el prunning permite descartar trials que no se ve que mejoraran el resultado guardado hasta ese entonces como el mejor. Al evitar el entrenamiento completo de modelos con configuraciones subóptimas, el optimizador puede explorar un mayor número de trials dentro del mismo límite de tiempo y alcanzar un resultado similar/mejor en el mismo tiempo.

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [50]:
# Inserte su código acá

from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_param_importances

# 1. Gráfico de historial de optimización
fig_history = plot_optimization_history(study_prune)
fig_history.update_layout(xaxis = dict(range=[-5, len(study_prune.trials)], tickvals=list(range(0, len(study_prune.trials) + 1, 10))))
fig_history.show()

# 2. Gráfico de coordenadas paralelas
fig_parallel = plot_parallel_coordinate(study_prune)
fig_parallel.show()

# 3. Gráfico de importancia de hiperparámetros
fig_importance = plot_param_importances(study_prune)
fig_importance.show()

Observando el gráfico de historial de optimización, se puede ver que las mejoras más significativas ocurren en los primeros trials (casi cercano a 0), bajando drasticamente sus valor (en este sentido, obteniendo un mejor valor de MAE). Luego, se ve que los MAE obtenidos a los largo de los trials "siguen la tendencia", viendose puntos bastante cercanos al mejor MAE encontrado a lo largo de los trials

En el gráfico de coordenadas paralelas, las líneas conectan las combinaciones de hiperparámetros con sus respectivos valores de MAE. Las líneas que corresponden a MAE más bajos, las mas oscuras, tienden a agruparse en learning_rate (entre 0.1 y 0.09) y en min_frequency (menor a 0.1). Un poco mas disperso, pero si con una tendencia clara, encontramos a n_estimators (mayor a 600) y reg_alpha (menor a 0.2). El resto de hiperparámetros podemos ver que son variados sus valores que permiten obtener un bajo valor de MAE

A partir del gráfico de importancia de hiperparámetros, podemos ver una clara dominancia de min_frequency con un 0.83, siendo el mas importante por mucha diferencia, seguido por min_child_weight con un 0.06 y reg_alpha con un 0.05.

Para hacer la comparativa, se plotea tambien para el casos sin prunning, mas que nada ver como se comportan las graficas para los 2 metodos

In [51]:
# 1. Historial de optimización sin pruning
fig1 = plot_optimization_history(study)
fig1.update_layout(xaxis = dict(range=[-5, len(study.trials)], tickvals=list(range(0, len(study.trials) + 1, 10))))
fig1.show()

# 2. Coordenadas paralelas sin pruning
fig2 = plot_parallel_coordinate(study)
fig2.show()

# 3. Gráfico de importancia de hiperparámetros sin pruning
fig3 = plot_param_importances(study)
fig3.show()

A diferencia del caso anterior, si bien el mejor MAE tiene como el "mismo comportamiento", se observa que los MAE obtenidos a los largo de los trials varian ampliamente en la grafica, viendose puntos bastante cercanos y muy lejanos (incluso mas alto que el inicial) al mejor MAE encontrado a lo largo de los trials.

En el gráfico de coordenadas paralelas, vemos que la tendencia de learning_rate se pierde, pero se mantiene en min_frequency (menor a 0.1), y se observan combinaciones mas variadas, que no siguen una unica tendencia, en el resto de hiperparámetros, dificultando poder obtener una idea de las mejores combinaciones, como lo hicimos en el caso anterior.

Finalmente, en el gráfico de importancia de hiperparámetros, podemos ver que la dominancia de min_frequency aqui es casi total con un 0.99, dando a entender que el resto de hiperparámetros eran minimamente (por no decir nada) relevantes en la optimizacion.

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [62]:
# Inserte su código acá

# 1. Genere una tabla resumen del MAE en el conjunto de validación
results = {
    "DummyRegressor": mae_dummy,
    "XGBoost": mae_xgb,
    "XGBoost (Constraints)": mae_constrained,
    "XGBoost (Optuna)": study.best_value,
    "XGBoost (Optuna + Prunning)": study_prune.best_value
}

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['MAE (Validación)'])
results_df = results_df.sort_values(by='MAE (Validación)')
results_df['MAE (Validación)'] = results_df['MAE (Validación)'].round(2)
display(results_df)

,MAE (Validación)
XGBoost (Optuna + Prunning),1961.73
XGBoost (Optuna),1987.30
XGBoost,2445.65
XGBoost (Constraints),2459.38
DummyRegressor,13408.36


El modelo que obtiene el mejor rendimiento, o bien el MAE mas bajo, es el de Optuna con Prunning, gracias a que evita el entrenamiento completo de modelos con configuraciones subóptimas, alcanzando un modelo con un rendimiento predictivo superio.

In [64]:
# 3. Cargue el mejor modelo, prediga sobre el conjunto de test y reporte su MAE
best_model_path = "best_xgb_modelPRU.pkl"
loaded_best_model = joblib.load(best_model_path)

preprocessor_loaded, booster_loaded = loaded_best_model
X_test_transformed = date_transformer.transform(X_test)
X_test_transformed = preprocessor_loaded.transform(X_test_transformed)
dtest = xgb.DMatrix(X_test_transformed)
y_pred_test = booster_loaded.predict(dtest)

mae_test = mean_absolute_error(y_test, y_pred_test)
print(f"\nMAE en el conjunto de test: {mae_test:.2f}")
print(f"\nMAE en el conjunto de validación: {study_prune.best_value:.2f}")


MAE en el conjunto de test: 1943.27

MAE en el conjunto de validación: 1961.73


MAE en el conjunto de test mejora levemente con respecto al conjunto de validación, indicando que el modelo generaliza bien y logrando que su desempeño en datos no vistos no empeore. Esto puede ocurrir debido a que los conjuntos son muestras diferentes de los datos, pudiendo tener ligeras diferencias en sus distribuciones. (o incluso la misma semilla que seteamos puede inducir leves variaciones en los resultados obtenidos, pero dudamos que sea demasiado el efecto como para que vayan a cambiar las conclusiones)


# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>